In [1]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("C:/IDEAL_Programming")
DATA_PATH = BASE_DIR / "processed" / "final" / "IDEAL_final_hourly_dataset.csv"
OUT_PATH = BASE_DIR / "processed" / "stores" / "history_store.csv"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# 1) διάλεξε ένα σπίτι που ξέρεις ότι έχει καλό coverage
HOME_ID = "home100" # άλλαξέ το αν θες
SOURCE_YEAR = 2017
TARGET_YEAR = 2026

usecols = ["home_id", "timestamp", "consumption_Wh"]
df = pd.read_csv(DATA_PATH, usecols=usecols, parse_dates=["timestamp"], low_memory=False)
df = df.dropna(subset=["home_id", "timestamp", "consumption_Wh"]).copy()
df["home_id"] = df["home_id"].astype(str)

# κρατάμε το home + year
src = df[(df["home_id"] == HOME_ID) & (df["timestamp"].dt.year == SOURCE_YEAR)].copy()
src = src.sort_values("timestamp")

# κρατάμε μόνο full-year hours (θέλουμε 8760)
# αν έχει παραπάνω/λιγότερο, απλά παίρνουμε το intersection 2017-01-01..2018-01-01
start = pd.Timestamp(f"{SOURCE_YEAR}-01-01 00:00:00")
end = pd.Timestamp(f"{SOURCE_YEAR+1}-01-01 00:00:00")
src = src[(src["timestamp"] >= start) & (src["timestamp"] < end)].copy()

# optional: αν λείπουν ώρες, κάνε reindex σε ωριαίο και fill (πιο “τέλειο” demo)
full_idx = pd.date_range(start=start, end=end, freq="h", inclusive="left")
src = src.set_index("timestamp").reindex(full_idx)
src.index.name = "timestamp"
src["consumption_Wh"] = pd.to_numeric(src["consumption_Wh"], errors="coerce")

# fill missing (forward fill, μετά backfill για αρχή)
src["consumption_Wh"] = src["consumption_Wh"].ffill().bfill()

# map timestamps to target year (ίδιο month/day/hour)
ts = src.index
mapped = [pd.Timestamp(year=TARGET_YEAR, month=t.month, day=t.day, hour=t.hour) for t in ts]

out = pd.DataFrame({
    "timestamp": mapped,
    "consumption_Wh": src["consumption_Wh"].astype(float).values
}).sort_values("timestamp")

# sanity: κρατάμε μόνο 2026
out = out[(out["timestamp"] >= pd.Timestamp("2026-01-01")) & (out["timestamp"] < pd.Timestamp("2027-01-01"))]
out.to_csv(OUT_PATH, index=False, encoding="utf-8")

print("Saved:", OUT_PATH)
print("Rows:", len(out), "| From:", out["timestamp"].min(), "To:", out["timestamp"].max())


Saved: C:\IDEAL_Programming\processed\stores\history_store.csv
Rows: 8760 | From: 2026-01-01 00:00:00 To: 2026-12-31 23:00:00
